# WandaSP / Magnitude Pruning Evaluation

## Set Up

In [ ]:
!pip install -q transformers==4.48.0 datasets wandb sentencepiece accelerate lm-eval -U

In [ ]:
import os
import random
from types import SimpleNamespace

import numpy as np
import torch
from transformers import AutoTokenizer, logging
from lm_eval import simple_evaluate
from lm_eval.models.huggingface import HFLM

from lib.prune import prune_magnitude_sp, prune_wanda_sp, check_sparsity
from models.hf_llama.modeling_llama import LlamaForCausalLM

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
logging.set_verbosity_error()

In [ ]:
SEED: int = 1
PRUNING_RATIO: float = 0.5
MODEL_NAME: str = "huggyllama/llama-7b"
CACHE_DIR: str = "llm_weights"
DEVICE: str = "cuda:0"
CALIB_DATASET: str = "wikitext2"
CALIB_NSAMPLES: int = 2048
EVAL_BATCH_SIZE: int = 64
TASKS = ["boolq", "arc_easy", "arc_challenge", "winogrande", "hellaswag"]

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
def get_llm(model_name: str, cache_dir: str = CACHE_DIR):
    model = LlamaForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        cache_dir=cache_dir,
        low_cpu_mem_usage=True,
    )
    for layer in model.model.layers:
        layer.self_attn.o_proj.bias = torch.nn.Parameter(torch.zeros_like(layer.self_attn.o_proj.bias, device="cpu"))
        layer.mlp.down_proj.bias = torch.nn.Parameter(torch.zeros_like(layer.mlp.down_proj.bias, device="cpu"))
        torch.nn.init.zeros_(layer.self_attn.o_proj.bias)
        torch.nn.init.zeros_(layer.mlp.down_proj.bias)
    model.seqlen = 128
    return model

In [ ]:
def make_prune_args(*, pruning_ratio: float) -> SimpleNamespace:
    return SimpleNamespace(
        pruning_ratio=pruning_ratio,
        nsamples=CALIB_NSAMPLES,
        seed=SEED,
        calibration_dataset=CALIB_DATASET,
        wanda_calib_dataset=CALIB_DATASET,
        remove_heads=-1,
        unstr=False,
    )


def load_fresh_model_and_tokenizer() -> tuple[torch.nn.Module, AutoTokenizer, torch.device]:
    model = get_llm(MODEL_NAME)
    device = torch.device(DEVICE if torch.cuda.is_available() else "cpu")
    model.to(device).eval()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    return model, tokenizer, device


def run_eval(model: torch.nn.Module, tokenizer: AutoTokenizer, title: str) -> dict:
    lm = HFLM(pretrained=model, tokenizer=tokenizer, batch_size=EVAL_BATCH_SIZE)
    results = simple_evaluate(model=lm, tasks=TASKS, batch_size=EVAL_BATCH_SIZE, limit=None)
    print(f"\n=== {title} ===")
    for task, metrics in results.get("results", {}).items():
        print(f"\n{task}:")
        for key, value in metrics.items():
            if key != "alias" and isinstance(value, (int, float)):
                print(f"  {key:28s} {value:.4f}")
            elif key != "alias":
                print(f"  {key:28s} {value}")
    return results

## Magnitude Pruning

In [ ]:
model, tokenizer, device = load_fresh_model_and_tokenizer()
args = make_prune_args(pruning_ratio=PRUNING_RATIO)
prune_magnitude_sp(args, model, tokenizer, device)
print(f"sparsity sanity check: {check_sparsity(model):.4f}")
magnitude_results = run_eval(model, tokenizer, "Magnitude Pruning")

## WandaSP Pruning

In [ ]:
model, tokenizer, device = load_fresh_model_and_tokenizer()
wandasp_args = make_prune_args(pruning_ratio=PRUNING_RATIO)
prune_wanda_sp(wandasp_args, model, tokenizer, device)
print(f"sparsity sanity check: {check_sparsity(model):.4f}")
wandasp_results = run_eval(model, tokenizer, "WandaSP Pruning")